# Lab 6 · Is the difference real?

**What you'll build:** the paired bootstrap confidence interval, and the
reasoning that turns two nearly-equal RMSEs into a defensible verdict.

You will finish this notebook holding the project's headline number.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labgrader import panel, grade_lab, TARGET, CLIMATE

df = panel()
print(df.shape, "rows x columns")
print("target column:", TARGET)

## 1. Run both models — the real ones

You have now hand-built every piece of this. So for the final run, call the
actual pipeline rather than a lookalike: these are the functions that produced
the numbers in the preprint, and you should recognise all of them.

- `experiment.upscale` — your Lab 5 assignment.
- `experiment.spatial_blocks` — your Lab 3 assignment.
- `experiment.blocked_cv` — your Lab 3 assignment, wrapped in 5 re-seeded repeats.

Takes about 15 seconds. The model is a random forest rather than the linear fits
you have been writing; nothing about the reasoning below depends on that.

In [ ]:
from src.pipeline import experiment as ex
from src.pipeline.assemble import predictor_columns

FEATS = predictor_columns(df)["climate_only"]
print(f"{len(FEATS)} climate-only predictors — no satellite variables, or the")
print("target would be predicted from a rearrangement of itself.\n")

dfB = ex.upscale(df, FEATS, cell_m=4000.0)

folds_a, _ = ex.blocked_cv(df,  FEATS, n_blocks=5, n_repeats=5)
folds_b, _ = ex.blocked_cv(dfB, FEATS, n_blocks=5, n_repeats=5)

print(f"Model A (scale-free) : RMSE {folds_a.rmse.mean():.5f}   R2 {folds_a.r2.mean():+.3f}")
print(f"Model B (coarse 4km) : RMSE {folds_b.rmse.mean():.5f}   R2 {folds_b.r2.mean():+.3f}")
print(f"folds: {len(folds_a)}  (5 blocks x 5 re-seeded repeats)")

> Both R² are **negative** — both models are worse than predicting the mean.
> Hold that thought; it decides the verdict in section 3.

In [ ]:
# Pair the folds. The merge on (repeat, block) is what makes the difference
# paired rather than two independent samples.
merged = folds_a[["repeat", "block", "rmse"]].merge(
    folds_b[["repeat", "block", "rmse"]], on=["repeat", "block"], suffixes=("_a", "_b"))
d = (merged["rmse_a"] - merged["rmse_b"]).to_numpy()

print(f"{len(d)} paired folds")
print(f"mean difference (A - B): {d.mean():+.6f}   (negative favours A)")

## 2. Why paired, and why bootstrap

**Paired.** Fold 3 might be a genuinely hard part of Surrey — both models do
badly there. If you compared the two lists of RMSEs as independent samples, that
fold-to-fold difficulty would swamp the small A-vs-B contrast. Subtract *within
each fold* and the shared difficulty cancels.

**Bootstrap, not a t-test.** A t-interval assumes the differences are normally
distributed. You have 25 folds, from 5 re-seeded partitions of the same 153
polygons — nowhere near enough to trust that assumption. The bootstrap makes no
assumption about shape: resample the differences with replacement thousands of
times, and read the interval off the spread of the resampled means.

### ✏️ Assignment 1 — `paired_bootstrap`

Return `(ci_lo, ci_hi)`, the 2.5th and 97.5th percentiles.

Use exactly this construction so your interval reproduces the published one:

```python
rng  = np.random.default_rng(seed)
boot = rng.choice(d, size=(n_boot, len(d)), replace=True).mean(axis=1)
```

Each row of `boot` is one resampled set of folds; its mean is one plausible value
for the true difference. `np.percentile` on those gives the interval.

In [ ]:
def paired_bootstrap(d, n_boot=10000, seed=26910):
    """95% bootstrap CI for the mean of the paired differences `d`."""
    # >>> YOUR TURN
    raise NotImplementedError

In [ ]:
lo, hi = paired_bootstrap(d)
print(f"mean paired difference (A - B): {d.mean():+.6f}")
print(f"95% CI: [{lo:+.6f}, {hi:+.6f}]")
print(f"A better in {(d < 0).mean()*100:.0f}% of folds")

# The pipeline's own version, for comparison. It should match to the digit.
ref = ex.paired_difference(folds_a, folds_b, "rmse")
print(f"\nexperiment.paired_difference: {ref['mean_diff']:+.6f} "
      f"[{ref['ci_lo']:+.6f}, {ref['ci_hi']:+.6f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rng = np.random.default_rng(26910)
ax.hist(rng.choice(d, size=(10000, len(d)), replace=True).mean(axis=1),
        bins=60, color="#8899aa")
ax.axvline(0, color="crimson", lw=2, label="no difference")
ax.axvline(lo, color="k", ls="--"); ax.axvline(hi, color="k", ls="--")
ax.set_xlabel("mean RMSE difference (A - B)"); ax.legend()
ax.set_title("10,000 resampled worlds. Zero sits inside the interval.")
plt.show()

### ✏️ Assignment 2 — `excludes_zero`

A confidence interval answers one question: *is zero inside it?*

- Interval entirely below zero → A is significantly better.
- Interval entirely above zero → B is significantly better.
- Interval **straddles** zero → you cannot tell them apart.

In [ ]:
def excludes_zero(lo, hi):
    """True if the whole interval sits on one side of zero."""
    # >>> YOUR TURN
    raise NotImplementedError


print("your interval excludes zero:", excludes_zero(lo, hi))

## 3. The verdict — and the trap in calling it

The cells above should have reproduced the preprint's headline numbers exactly:

| | |
|---|---|
| Model A | RMSE 0.01393, R² −0.042 |
| Model B | RMSE 0.01397, R² −0.051 |
| paired ΔRMSE | −0.00004, CI **[−0.00016, +0.00007]** |

The interval straddles zero. The tempting conclusion is *"the hypothesis is not
supported — downscaling doesn't help."*

**That conclusion would be wrong**, and the reason is the most sophisticated
thing in this project.

"Not supported" is a claim *about downscaling*. To make it, the experiment has to
have been capable of detecting a difference in the first place. Two preconditions:

1. **Skill.** Do the models predict anything at all? Both R² are **negative** —
   worse than predicting the mean. Ranking two failures tells you nothing about
   why they failed. *(Lab 1 showed you why: the predictors vary in time, the
   target varies in space.)*

2. **Contrast.** Did coarsening actually change the predictors? It removed ~12%
   of spatial variance, against a 30% gate. A and B are nearly the same model, so
   of course their difference is zero. *(Lab 5.)*

Both fail. So the honest verdict is **INCONCLUSIVE** — the experiment reporting
that, as specified over Surrey, it had nothing to measure. That is a different
claim from the hypothesis being wrong, and `experiment.verdict()` enforces the
distinction in code so that no future run can quietly skip it.

The Fraser Valley transect is what clears the gates: 48% variance removed at
25 km, and there the paired CI *does* exclude zero — in Model B's favour. Which
is why the project's real verdict is **FALSIFIED at Extent 2**, and why the
preprint's title says *"resolution was not the lever."*

### ✏️ Assignment 3 — `MY_VERDICT`

One word, from the reasoning above.

In [ ]:
MY_VERDICT = ""   # >>> YOUR TURN

In [ ]:
# And the pipeline's own call, with the reasoning it prints. Compare it to yours.
call, why = ex.verdict(
    ex.paired_difference(folds_a, folds_b, "rmse"),
    ex.paired_difference(folds_a, folds_b, "r2"),
    r2_a=float(folds_a.r2.mean()), r2_b=float(folds_b.r2.mean()),
    var_removed=float(ex.upscaling_diagnostics(df, FEATS).var_removed_frac.median()),
)
print(call, "\n"); print(why)

---
## Grade it

In [ ]:
grade_lab(6, globals())

### What you should be able to say out loud

- Pairing over identical folds cancels fold difficulty; bootstrapping avoids a
  normality assumption 25 folds cannot support.
- A CI that straddles zero means *cannot distinguish* — not *proven equal*.
- A null is only evidence **if the experiment could have detected a difference**.
  Skill and contrast are checked first, in code.
- Surrey = INCONCLUSIVE. The transect = FALSIFIED. Different extents, different
  claims, one codebase.

### You're done

You have now rebuilt, in miniature, the whole spine of `src/pipeline/experiment.py`:
the target, the loss, honest folds, regularisation, the upscale, and the paired
interval. Open that file — it should read as familiar now, not as magic.